# Export Frozen V3 CatBoost Model -- Artifacts for the O7 Prototype

**Purpose:** No trained model artifact has existed anywhere in this project until now --
every notebook re-fits in memory and discards the model when the runtime disconnects.
This notebook fits the exact same frozen V3 CatBoost configuration used throughout
this thesis (identical to `HSLS_catboost_experiment_SetD (2).ipynb` and the FINAL SHAP
notebook -- copied verbatim, not reconstructed from memory) and saves the artifacts
the O7 prototype needs to make a real prediction without retraining:

- `catboost_v3_setD.cbm` -- the fitted CatBoost model, CatBoost's own native format
  (no cross-version pickle/joblib compatibility risk).
- `preprocessing_pipeline.joblib` -- the fitted preprocessing steps (impute -> scale)
  ONLY, fit on the training fold, exactly as used to produce this thesis's reported
  results. SMOTE is intentionally NOT included here -- SMOTE only ever resamples
  training data and must never be applied to a new student's input at prediction time.
- `feature_names.json` -- the exact 377-column order the model expects, so the
  prototype can build a correctly-ordered input row.
- `demo_students.csv` -- a sample of REAL held-out test-set students (predictors +
  true label), for the prototype's "pick an existing student" mode. No
  student-identifying columns are included -- this is Set D's own audited predictor
  set, which excludes identifiers by construction.
- `feature_defaults.json` -- per-feature median (continuous) / mode (discrete) values
  computed from the full TRAINING set, for the prototype's manual-entry mode, so
  unadjusted sliders reflect the real training population rather than the small demo
  sample.

**This is a one-time export step, not a new experiment.** No modelling decision here
differs from the frozen V3 CatBoost protocol already reported in the thesis.

## (Optional) Mount Google Drive

In [13]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Not running in Colab -- skipping Drive mount (expected for local VS Code runs).")


Not running in Colab -- skipping Drive mount (expected for local VS Code runs).


## Install / Import

In [14]:
!pip install imbalanced-learn --quiet
!pip install catboost --quiet

import os
import json as jsonlib
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

from catboost import CatBoostClassifier
from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SkPipeline


### CONFIG -- identical paths/settings to the frozen SHAP notebook

In [15]:
DATA_PATH      = "prepared_hsls_setD_4Y.csv"
TARGET_COL     = "program_stream"
COLS_TO_REMOVE = []
RANDOM_STATE   = 42

OUTPUT_DIR = "."
def out_path(filename: str) -> str:
    return f"{OUTPUT_DIR.rstrip('/')}/{filename}"

CATBOOST_DEFAULTS = dict(
    iterations=500, depth=6, learning_rate=0.1, l2_leaf_reg=3.0,
    early_stopping_rounds=30, val_size=0.15, random_state=RANDOM_STATE, verbose=False,
)


### SHARED: preprocessing helpers, CatBoost wrapper

Copied verbatim from `HSLS_catboost_experiment_SetD (2).ipynb` -- identical
`CatBoostEarlyStop`, `determine_column_types`, `build_preprocessor`, `make_cv`,
`smote_k_for`. Nothing here is reconstructed from memory.

In [16]:


def load_data(path: str, target_col: str, cols_to_remove: list) -> pd.DataFrame:
    df = pd.read_csv(path)
    present = [c for c in cols_to_remove if c in df.columns]
    if present:
        df = df.drop(columns=present)
    if df[target_col].isnull().sum() > 0:
        before = len(df)
        df = df.dropna(subset=[target_col]).reset_index(drop=True)
        print(f"Dropped {before - len(df)} rows with missing target.")
    print(f"Loaded: {df.shape[0]} rows x {df.shape[1]} columns")
    return df


def diagnose(f1_train, f1_test):
    gap = f1_train - f1_test
    if gap > 0.15:
        return gap, "OVERFIT"
    elif f1_test < 0.50 and f1_train < 0.50:
        return gap, "UNDERFIT"
    return gap, "GOOD FIT"


def evaluate(model, X_train, y_train, X_test, y_test, cv, name, extra=None):
    y_pred = model.predict(X_test)
    y_train_pred = model.predict(X_train)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
    rec = recall_score(y_test, y_pred, average="macro", zero_division=0)
    f1_test = f1_score(y_test, y_pred, average="macro", zero_division=0)
    f1_train = f1_score(y_train, y_train_pred, average="macro", zero_division=0)
    gap, diag = diagnose(f1_train, f1_test)

    cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1_macro", n_jobs=-1)

    row = {
        "Model": name,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1 Test": round(f1_test, 4),
        "F1 Train": round(f1_train, 4),
        "Train-Test Gap": round(gap, 4),
        "CV Mean": round(cv_scores.mean(), 4),
        "CV Std": round(cv_scores.std(), 4),
        "Diagnosis": diag,
    }
    if extra:
        row.update(extra)
    return row


class CatBoostEarlyStop(BaseEstimator, ClassifierMixin):
    """CatBoost wrapper with SELF-CONTAINED early stopping.

    Carves its own train/validation split out of whatever X, y it is given at .fit()
    time -- and .fit() is only ever called with TRAINING data (a pipeline's training
    fold, a CV fold's training portion, or GridSearchCV's per-fold training split). It
    never receives the outer held-out test set, so the test set can never leak into
    early stopping -- no fit_params plumbing through Pipeline/GridSearchCV/
    cross_val_score is needed, because the validation carve-out happens INSIDE this
    estimator, transparently, every place it is fit.

    Exposes the standard sklearn estimator API (get_params/set_params via
    BaseEstimator, fit/predict/predict_proba) so it drops into the existing
    ImbPipeline/SkPipeline/GridSearchCV/cross_val_score call sites with no other code
    changes required in V1-V6.
    """

    def __init__(self, iterations=500, depth=6, learning_rate=0.1,
                 l2_leaf_reg=3.0, early_stopping_rounds=30,
                 val_size=0.15, random_state=42, verbose=False):
        self.iterations = iterations
        self.depth = depth
        self.learning_rate = learning_rate
        self.l2_leaf_reg = l2_leaf_reg
        self.early_stopping_rounds = early_stopping_rounds
        self.val_size = val_size
        self.random_state = random_state
        self.verbose = verbose

    def _build_model(self):
        return CatBoostClassifier(
            iterations=self.iterations,
            depth=self.depth,
            learning_rate=self.learning_rate,
            l2_leaf_reg=self.l2_leaf_reg,
            random_state=self.random_state,
            verbose=self.verbose,
            loss_function="MultiClass",
        )

    def fit(self, X, y):
        y = np.asarray(y)
        min_class_count = pd.Series(y).value_counts().min()
        try:
            if min_class_count < 2:
                raise ValueError("class too small to stratify")
            X_tr, X_val, y_tr, y_val = train_test_split(
                X, y, test_size=self.val_size, random_state=self.random_state, stratify=y
            )
        except ValueError:
            X_tr, X_val, y_tr, y_val = train_test_split(
                X, y, test_size=self.val_size, random_state=self.random_state
            )

        self.model_ = self._build_model()
        self.model_.fit(
            X_tr, y_tr,
            eval_set=(X_val, y_val),
            early_stopping_rounds=self.early_stopping_rounds,
            use_best_model=True,
            verbose=self.verbose,
        )
        self.classes_ = self.model_.classes_
        return self

    def predict(self, X):
        return np.asarray(self.model_.predict(X)).ravel()

    def predict_proba(self, X):
        return self.model_.predict_proba(X)


# BASE_MODEL_DEFS trimmed: this notebook fits the frozen V3 CatBoost only, to
# export it -- not a five-model comparison.
def make_frozen_catboost():
    return CatBoostEarlyStop(**CATBOOST_DEFAULTS)
# CHANGED FROM OFFICIAL PIPELINE: "SVM": SVC(...) is replaced by "CatBoost":
# CatBoostEarlyStop(...) above. Decision Tree, Random Forest, KNN, Naive Bayes
# are byte-for-byte identical to the official file.


def smote_k_for(y_train):
    min_class_count = pd.Series(y_train).value_counts().min()
    return max(1, min(5, min_class_count - 1))


def make_cv(y):
    """StratifiedKFold with n_splits capped by the smallest class's size, not
    hardcoded to 5. A class with only 2-3 members (common in imbalanced multi-class
    data) can't support 5-fold CV; this adapts instead of silently erroring deep
    inside sklearn or, worse, running with folds that don't actually contain every
    class."""
    min_class_count = pd.Series(y).value_counts().min()
    n_splits = min(5, int(min_class_count))
    if n_splits < 2:
        raise ValueError(
            f"Cross-validation impossible: smallest class has only "
            f"{min_class_count} sample(s), need at least 2 for any CV split."
        )
    if n_splits < 5:
        print(f"  NOTE: smallest class has {min_class_count} samples -- "
              f"using {n_splits}-fold CV instead of the default 5-fold.")
    return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)


class ColumnIndexSelector(BaseEstimator, TransformerMixin):
    """Selects fixed column positions from an already-preprocessed array. Used by V4
    to apply its one-time feature ranking (chi2/RFE/RF-importance, computed once --
    exploratory, matches the original V4 design) inside a pipeline whose
    IMPUTATION/ENCODING/SCALING still gets refit fresh per CV fold. Only the selected
    feature *positions* are fixed in advance; the preprocessing that produces those
    positions' values is not."""
    def __init__(self, indices):
        self.indices = indices
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return X[:, self.indices]


def determine_column_types(df_train_raw: pd.DataFrame, target_col: str):
    """Classifies each predictor as continuous/discrete/categorical from the training
    schema. This is a one-time STRUCTURAL decision, not a fold-sensitive statistic --
    safe to do once outside CV."""
    numeric_cols = df_train_raw.select_dtypes(include="number").columns.tolist()
    categorical_cols = [c for c in df_train_raw.select_dtypes(include="object").columns if c != target_col]
    continuous_cols, discrete_cols = [], []
    for col in numeric_cols:
        if col == target_col:
            continue
        non_null = df_train_raw[col].dropna()
        is_integer = non_null.apply(lambda x: x == int(x)).all() if len(non_null) else True
        n_unique = df_train_raw[col].nunique()
        if is_integer and n_unique <= 20:
            discrete_cols.append(col)
        else:
            continuous_cols.append(col)
    return continuous_cols, discrete_cols, categorical_cols


def build_preprocessor(continuous_cols, discrete_cols, categorical_cols):
    """Unfitted ColumnTransformer: mean-impute continuous, median-impute discrete,
    mode-impute + ordinal-encode categorical (unseen categories map to -1). Nothing is
    fit here -- fitting happens fresh each time this is cloned into a Pipeline and
    that Pipeline is fit, which is what makes this fold-safe."""
    continuous_pipe = SkPipeline([("impute", SimpleImputer(strategy="mean"))])
    discrete_pipe = SkPipeline([("impute", SimpleImputer(strategy="median"))])
    categorical_pipe = SkPipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ])
    return ColumnTransformer(transformers=[
        ("cont", continuous_pipe, continuous_cols),
        ("disc", discrete_pipe, discrete_cols),
        ("cat", categorical_pipe, categorical_cols),
    ])


## Load data, prepare (fold-safe), and fit the frozen V3 pipeline

Identical logic to `prepare_v2_style` + `fit_frozen_v3_catboost` in the SHAP FINAL
notebook: split before any preprocessing, fit imputation/scaling on the training fold
only, SMOTE applied only during `.fit()` (never at prediction time).

In [17]:
df = pd.read_csv(DATA_PATH)
for col in COLS_TO_REMOVE:
    if col in df.columns:
        df = df.drop(columns=[col])

d = df.dropna(subset=[TARGET_COL]).copy()
train_idx, test_idx = train_test_split(
    d.index, test_size=0.2, random_state=RANDOM_STATE, stratify=d[TARGET_COL]
)
df_train_raw = d.loc[train_idx].copy()
df_test_raw = d.loc[test_idx].copy()

continuous_cols, discrete_cols, categorical_cols = determine_column_types(df_train_raw, TARGET_COL)
feature_names = continuous_cols + discrete_cols + categorical_cols
print(f"Feature count: {len(feature_names)} (expected 377)")

X_train_raw = df_train_raw[feature_names]
X_test_raw = df_test_raw[feature_names]
y_train = df_train_raw[TARGET_COL].astype(int).values
y_test = df_test_raw[TARGET_COL].astype(int).values

smote_k = smote_k_for(y_train)
preprocessor = build_preprocessor(continuous_cols, discrete_cols, categorical_cols)

pipe = ImbPipeline([
    ("preprocess", preprocessor),
    ("scale", MinMaxScaler()),
    ("smote", SMOTE(random_state=RANDOM_STATE, k_neighbors=smote_k)),
    ("model", make_frozen_catboost()),
])
pipe.fit(X_train_raw, y_train)

y_pred_test = pipe.predict(X_test_raw)
f1_test = f1_score(y_test, y_pred_test, average="macro", zero_division=0)
acc = accuracy_score(y_test, y_pred_test)
print(f"Reproduced test Macro-F1: {f1_test:.4f} (documented reference: 0.2287)")
print(f"Reproduced test Accuracy: {acc:.4f} (documented reference: 0.3137)")
if abs(f1_test - 0.2287) > 0.01:
    print("*** WARNING: this run's Macro-F1 differs from the documented reference by "
          "more than 0.01 -- check DATA_PATH and package versions before exporting. ***")
else:
    print("Within tolerance of the documented reference -- safe to export.")


Feature count: 377 (expected 377)
Reproduced test Macro-F1: 0.2287 (documented reference: 0.2287)
Reproduced test Accuracy: 0.3137 (documented reference: 0.3137)
Within tolerance of the documented reference -- safe to export.


## Export the artifacts

**SMOTE is deliberately excluded from what gets saved.** `pipe` above is an
imbalanced-learn `Pipeline`, which already only applies SMOTE during `.fit()` and
automatically skips it during `.predict()` -- but to keep the saved preprocessing
artifact simple, portable, and impossible to misuse, only the already-fitted
`preprocess` and `scale` steps are re-packaged into a plain scikit-learn `Pipeline`
(no sampler) before saving. The CatBoost model is saved via its own native format,
not pickled inside a custom wrapper class -- this avoids any dependency on
`CatBoostEarlyStop` being importable wherever the prototype runs.

In [18]:
from sklearn.pipeline import Pipeline as SkPipeline

# Native CatBoost model (the actual fitted classifier inside the CatBoostEarlyStop wrapper).
catboost_native_model = pipe.named_steps["model"].model_
catboost_native_model.save_model(out_path("catboost_v3_setD.cbm"))
print(f"Saved: {out_path('catboost_v3_setD.cbm')}")

# Preprocessing only (impute + scale) -- no sampler, safe to apply to a single new row.
preprocessing_pipeline = SkPipeline([
    ("preprocess", pipe.named_steps["preprocess"]),
    ("scale", pipe.named_steps["scale"]),
])
joblib.dump(preprocessing_pipeline, out_path("preprocessing_pipeline.joblib"))
print(f"Saved: {out_path('preprocessing_pipeline.joblib')}")

# Exact expected column order.
with open(out_path("feature_names.json"), "w") as f:
    jsonlib.dump(feature_names, f)
print(f"Saved: {out_path('feature_names.json')} ({len(feature_names)} columns)")

# A sample of real held-out test-set students, for the prototype's "pick an existing
# student" mode. Predictors + true label only -- Set D's audited predictor set already
# excludes student-identifying columns by construction.
rng = np.random.RandomState(RANDOM_STATE)
sample_size = min(40, len(X_test_raw))
sample_idx = rng.choice(X_test_raw.index, size=sample_size, replace=False)
demo_df = X_test_raw.loc[sample_idx].copy()
demo_df[TARGET_COL] = df_test_raw.loc[sample_idx, TARGET_COL].values
demo_df.to_csv(out_path("demo_students.csv"), index=False)
print(f"Saved: {out_path('demo_students.csv')} ({sample_size} real held-out students)")

# Per-feature default values (median for continuous, mode for discrete) computed from
# the FULL TRAINING SET -- not the small demo sample above -- for the prototype's
# manual-entry mode, so unadjusted sliders reflect the real training population.
feature_defaults = {}
for col in feature_names:
    series = X_train_raw[col].dropna()
    if col in continuous_cols:
        feature_defaults[col] = float(series.median()) if len(series) else 0.0
    else:
        feature_defaults[col] = float(series.mode().iloc[0]) if len(series) else 0.0
with open(out_path("feature_defaults.json"), "w") as f:
    jsonlib.dump(feature_defaults, f)
print(f"Saved: {out_path('feature_defaults.json')} (training-set medians/modes)")

print("\nAll O7 prototype artifacts exported. Reproduced Macro-F1 for this exported "
      f"model: {f1_test:.4f} (documented reference: 0.2287).")


Saved: ./catboost_v3_setD.cbm
Saved: ./preprocessing_pipeline.joblib
Saved: ./feature_names.json (377 columns)
Saved: ./demo_students.csv (40 real held-out students)
Saved: ./feature_defaults.json (training-set medians/modes)

All O7 prototype artifacts exported. Reproduced Macro-F1 for this exported model: 0.2287 (documented reference: 0.2287).
